# Case 4: dynamic latent debugger

In [ ]:
import sys; sys.path.insert(0, '../src')
import numpy as np
import matplotlib.pyplot as plt
from causal_opt.simulation import simulate_svar
from causal_opt.methods.dynamic_latent import fit_dynamic_latent
from causal_opt.evaluation.dynamic_metrics import dynamic_metrics
from causal_opt.evaluation.latent_metrics import latent_metrics
latent_process='iid'
data=simulate_svar(300,5,1,1,s0=5,k=2,burn_in=100,lag_sparsity=.25,latent_process=latent_process,min_latent_children=2)
print(data.X.shape,'radius=',data.spectral_radius,'condition=',data.condition_number)

In [ ]:
fig,ax=plt.subplots(2,3,figsize=(13,6)); ax[0,0].plot(data.X[:100,:3]); ax[0,0].set_title('Observed series'); ax[0,1].imshow(data.W0_true,cmap='coolwarm'); ax[0,1].set_title('True W0'); ax[0,2].imshow(data.W_lags_true[0],cmap='coolwarm'); ax[0,2].set_title('True W lag 1'); ax[1,0].imshow(data.L_true,cmap='coolwarm'); ax[1,0].set_title('True L'); ax[1,1].imshow(data.C_true[:100],aspect='auto',cmap='coolwarm'); ax[1,1].set_title('C true window'); ax[1,2].plot(np.linalg.svd(data.C_true,compute_uv=False),'o-'); ax[1,2].set_title('C true singular values'); plt.tight_layout()

In [ ]:
ours=fit_dynamic_latent(data.X,1,k=2,lambda_0=.1,lambda_lag=.005,lambda_latent=.05,wlag_threshold=.05,max_outer_iter=30,inner_max_iter=400,random_state=1)
print(dynamic_metrics(data.W0_true,data.W_lags_true,ours.W0,ours.W_lags,data.X,w0_threshold=0,wlag_threshold=0)); print(latent_metrics(data.C_true,ours.C,data.L_true,ours.L)); print('rank/norm/balance',ours.diagnostics['effective_rank_C'],ours.diagnostics['C_fro_norm'],ours.diagnostics['factor_balance_ratio'])

In [ ]:
fig,ax=plt.subplots(2,3,figsize=(13,6)); ax[0,0].imshow(ours.W0,cmap='coolwarm'); ax[0,0].set_title('Estimated W0'); ax[0,1].imshow(ours.W_lags[0],cmap='coolwarm'); ax[0,1].set_title('Estimated W lag 1'); ax[0,2].imshow(ours.C[:100],aspect='auto',cmap='coolwarm'); ax[0,2].set_title('C estimated window'); ax[1,0].plot(np.linalg.svd(data.C_true,compute_uv=False),'o-',label='true'); ax[1,0].plot(np.linalg.svd(ours.C,compute_uv=False),'o-',label='estimated'); ax[1,0].legend(); ax[1,0].set_title('C singular values'); h=ours.diagnostics['history']; ax[1,1].semilogy([max(r['h'],1e-18) for r in h]); ax[1,1].set_title('Acyclicity h(W0) across outer iterations'); ax[1,2].plot([r['fit_loss'] for r in h]); ax[1,2].set_title('Data-fit loss across augmented-Lagrangian outer iterations'); plt.tight_layout()

In [ ]:
try:
    from causal_opt.baselines.lpcmci import fit_lpcmci
    baseline=fit_lpcmci(data.X,tau_max=1); print(baseline.diagnostics); print(baseline.graph)
except ImportError as exc: print('LPCMCI unavailable:',exc)
print('Liégeois runs only when its MATLAB repository and documented entrypoint are configured.')